In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-4 ; m=1.0 ; a=2.0 ; b=1.0 ; k=10.0 ; g=9.81 ;      # elipse x = a cosθ, y = b sinθ  (a ≥ b) ; muelle k ; y hacia arriba
tf=8; N=int(tf/dt);

t=np.linspace(0,tf,N)


# Condiciones iniciales  (θ0, θ̇0)
def caso_peq(delta0):
    """Pequeñas oscilaciones alrededor del equilibrio inferior θ = −π/2:  θ(0) = −π/2 + δ0 ,  θ̇(0) = 0."""
    return dict(label=f'$\\delta_0={delta0}$', th=-np.pi/2 + delta0, thd=0.0)

# SITUACIÓN 1: movimiento cualquiera 
caso_general = dict(label='movimiento general', th=0.5, thd=-4.0)

# SITUACIÓN 2: pequeñas oscilaciones (apartado c). Los otros δ0 se usan para comparar con la solución analítica
casos = [caso_peq(0.1), caso_peq(0.025), caso_peq(0.05), caso_peq(0.2), caso_general]
i_peq, i_gen = 0, 4                                         # índices de las dos situaciones en 'casos'

## FUNCIONES GENERALES: LAGRANGIANO, HAMILTONIANO, EULER-LAGRANGE, HAMILTON Y PARÉNTESIS DE POISSON

In [ ]:
#FUNCIONES GENERALES (valen para cualquier sistema)
from types import SimpleNamespace

def lagrangian_hamiltonian(T, U, qs, qds, ps):
    """A partir de T y U devuelve:
         L = T − U                        (Lagrangiano)
         p_i = ∂L/∂q̇_i                    (momentos generalizados)
         H = Σ p_i q̇_i − L                (Hamiltoniano), SIEMPRE expresado en variables canónicas (q, p)
       T, U: expresiones de SymPy en (q, q̇) ;  qs, qds, ps: listas de símbolos (coordenadas, velocidades, momentos).
       Devuelve un objeto con L, p, H y, además, H_vel = H(q, q̇) y qdot = {q̇_i: q̇_i(q, p)}."""
    L = sp.simplify(T - U)
    p_expr = [sp.diff(L, qd) for qd in qds]                                           # p_i = ∂L/∂q̇_i  (derivadas parciales)
    H_vel  = sp.simplify(sum(p*qd for p, qd in zip(p_expr, qds)) - L)                 # H(q, q̇) = Σ p q̇ − L
    sol    = sp.solve([sp.Eq(P, pe) for P, pe in zip(ps, p_expr)], qds, dict=True)[0]  # se invierte p(q, q̇)  ->  q̇(q, p)
    H      = sp.simplify(H_vel.subs(sol))                                             # H(q, p)
    return SimpleNamespace(L=L, p=p_expr, H=H, H_vel=H_vel, qdot=sol)


def euler_lagrange(L, qs, qds, qdds, t=None):
    """Ecuaciones de Euler-Lagrange  d/dt(∂L/∂q̇_i) − ∂L/∂q_i = 0  para cualquier L(q, q̇ [, t]).
       qdds: símbolos para las aceleraciones q̈_i ; t: símbolo del tiempo si L depende explícitamente de él.
       Devuelve la lista de expresiones (cada una igualada a 0)."""
    n, EL = len(qs), []
    for i in range(n):
        p_i = sp.diff(L, qds[i])
        dp_dt = sum(sp.diff(p_i, qs[j])*qds[j] + sp.diff(p_i, qds[j])*qdds[j] for j in range(n))   # derivada total respecto a t
        if t is not None:
            dp_dt += sp.diff(p_i, t)
        EL.append(sp.simplify(dp_dt - sp.diff(L, qs[i])))
    return EL


def hamilton_equations(H, qs, ps):
    """Ecuaciones de Hamilton para cualquier H(q, p):  q̇_i = ∂H/∂p_i ,  ṗ_i = −∂H/∂q_i.  Devuelve (qdot_eq, pdot_eq)."""
    return [sp.diff(H, P) for P in ps], [-sp.diff(H, q) for q in qs]


def poisson_bracket(f, g, qs, ps):
    """Paréntesis de Poisson  {f, g} = Σ_i ( ∂f/∂q_i ∂g/∂p_i − ∂f/∂p_i ∂g/∂q_i )."""
    return sp.simplify(sum(sp.diff(f, q)*sp.diff(g, p) - sp.diff(f, p)*sp.diff(g, q) for q, p in zip(qs, ps)))

## (a) LAGRANGIANO, HAMILTONIANO Y ECUACIÓN DE MOVIMIENTO

In [ ]:
#(a) Lagrangiano, Hamiltoniano y ecuación de movimiento   (con las funciones generales)
th_s, thd_s, thdd_s, p_s = sp.symbols('theta thetadot thetaddot p', real=True)
m_, a_, b_, k_, g_ = sp.symbols('m a b k g', positive=True)

# Ligadura  x²/a² + y²/b² = 1 :  2 coordenadas (x, y) − 1 ligadura = 1 grado de libertad  ->  θ
x_s, y_s = a_*sp.cos(th_s), b_*sp.sin(th_s)
xd_s, yd_s = sp.diff(x_s, th_s)*thd_s, sp.diff(y_s, th_s)*thd_s

T_sym = sp.simplify(sp.Rational(1,2)*m_*(xd_s**2 + yd_s**2))
r2_s  = sp.simplify(x_s**2 + y_s**2)                                  # r² = distancia² al origen
U_sym = sp.simplify(sp.Rational(1,2)*k_*r2_s + m_*g_*y_s)            # muelle + gravedad
E_sym = T_sym + U_sym

lag = lagrangian_hamiltonian(T_sym, U_sym, [th_s], [thd_s], [p_s])
L_sym, p_expr, H_can = lag.L, lag.p, lag.H
print('T =', T_sym)
print('U =', U_sym)
print('L =', L_sym)
print('p =', p_expr)
print('H(θ, p) =', H_can)
print('H − E =', sp.simplify(lag.H_vel - E_sym), '   ->  H = E' if sp.simplify(lag.H_vel - E_sym) == 0 else '   ->  H ≠ E')
print('L no depende explícitamente de t (ligadura y fuerzas independientes del tiempo)  ->  H = E se conserva')

# Ecuación de Euler-Lagrange:  f1(θ) θ̈ + f2(θ) (k/m − θ̇²) + f3(θ) = 0
EL = euler_lagrange(L_sym, [th_s], [thd_s], [thdd_s])[0]
EL_m = sp.expand(EL/m_)
f1 = sp.simplify(sp.diff(EL_m, thdd_s))                              # coeficiente de θ̈
f2 = sp.simplify(-sp.diff(EL_m, thd_s, 2)/2)                         # coeficiente de θ̇² es −f2
f3 = sp.simplify(EL_m - f1*thdd_s - f2*(k_/m_ - thd_s**2))           # resto
print('\nf1(θ) =', f1)
print('f2(θ) =', f2)
print('f3(θ) =', f3)
print('comprobación  EL/m − [f1 θ̈ + f2 (k/m − θ̇²) + f3] =', sp.simplify(EL_m - (f1*thdd_s + f2*(k_/m_ - thd_s**2) + f3)))

# Ecuaciones de Hamilton
qdot_eq, pdot_eq = hamilton_equations(H_can, [th_s], [p_s])
print('\nθ̇ = ∂H/∂p =', qdot_eq[0])
pdot_compact = -sp.diff(U_sym, th_s) + p_s**2*sp.diff(f1, th_s)/(2*m_*f1**2)           # ṗ = −V'(θ) + p² μ'(θ) / (2 m μ²)
_chk = sp.lambdify((th_s, p_s, m_, a_, b_, k_, g_), pdot_eq[0] - pdot_compact, 'numpy')
print("ṗ = −∂H/∂θ = −V'(θ) + p² μ'(θ)/(2 m μ(θ)²)   (μ = f1)   ->  comprobación numérica:", abs(_chk(0.7, 1.3, 1.0, 2.0, 1.0, 10.0, 9.81)) < 1e-9)

par_s = (m_, a_, b_, k_, g_)
f_num   = sp.lambdify((th_s, p_s, *par_s), qdot_eq + pdot_eq, 'numpy', cse=True)     # (θ̇, ṗ)
H_num   = sp.lambdify((th_s, p_s, *par_s), H_can, 'numpy')
qd_num  = sp.lambdify((th_s, p_s, *par_s), lag.qdot[thd_s], 'numpy')
E_num   = sp.lambdify((th_s, thd_s, *par_s), E_sym, 'numpy')
L_num   = sp.lambdify((th_s, thd_s, *par_s), L_sym, 'numpy')
V_num   = sp.lambdify((th_s, *par_s), U_sym, 'numpy')
mu_num  = sp.lambdify((th_s, *par_s), f1, 'numpy')                                  # μ(θ) = a² sin²θ + b² cos²θ
d2V_num = sp.lambdify((th_s, *par_s), sp.diff(U_sym, th_s, 2), 'numpy')

# Paréntesis de Poisson: relación canónica y conservación
print('\n{θ, p} =', poisson_bracket(th_s, p_s, [th_s], [p_s]))
print('{θ, H} − θ̇ =', sp.simplify(poisson_bracket(th_s, H_can, [th_s], [p_s]) - qdot_eq[0]), ' ,  {p, H} − ṗ =', sp.simplify(poisson_bracket(p_s, H_can, [th_s], [p_s]) - pdot_eq[0]))
print('{H, H} =', poisson_bracket(H_can, H_can, [th_s], [p_s]), '  ->  H se conserva')
print('{p, H} ≠ 0  ->  p NO se conserva (θ no es cíclica)' if poisson_bracket(p_s, H_can, [th_s], [p_s]) != 0 else '{p, H} = 0')

## (b), (c) EQUILIBRIOS, ESTABILIDAD Y FRECUENCIA DE LAS PEQUEÑAS OSCILACIONES

In [ ]:
#(b), (c) Equilibrios, estabilidad y frecuencia de las pequeñas oscilaciones
V_sym  = U_sym
dV     = sp.diff(V_sym, th_s)
d2V    = sp.diff(V_sym, th_s, 2)
print("V'(θ)  =", sp.factor(dV))
print("V''(θ) =", sp.simplify(d2V))

print("\nθ = ±π/2 :  V'(±π/2) =", sp.simplify(dV.subs(th_s, sp.pi/2)), ',', sp.simplify(dV.subs(th_s, -sp.pi/2)), '  ->  son posiciones de equilibrio')
V2_top, V2_bot = sp.simplify(d2V.subs(th_s, sp.pi/2)), sp.simplify(d2V.subs(th_s, -sp.pi/2))
print("   V''(+π/2) =", V2_top, '   (estable si k(a²−b²) > m g b)')
print("   V''(−π/2) =", V2_bot, '   (>0 siempre, pues a ≥ b: estable)')

# Otro equilibrio:  V' = cosθ [ m g b − k (a²−b²) sinθ ] = 0   ->   sinθ_eq = m g b / (k (a²−b²))
s_eq = m_*g_*b_/(k_*(a_**2 - b_**2))
V2_eq = sp.simplify(d2V.subs(th_s, sp.asin(s_eq)))
print('\nOtro equilibrio:  sinθ_eq =', s_eq, '   (existe con |θ_eq| ≠ π/2  si  a > b  y  k(a²−b²) > m g b)')
print("   V''(θ_eq) =", V2_eq, '  = −k (a²−b²) cos²θ_eq  < 0  ->  siempre INESTABLE')

# Frecuencia de las pequeñas oscilaciones alrededor de θ = −π/2:   ω² = V''(−π/2) / (m μ(−π/2)),  μ(−π/2) = a²
om2_sym = sp.simplify(V2_bot/(m_*f1.subs(th_s, -sp.pi/2)))
print('\nω² =', om2_sym, '  =  g b/a² + (k/m)(1 − b²/a²)')
print('(i)  a = b :  ω² =', sp.simplify(om2_sym.subs(a_, b_)), '   ->  péndulo simple de longitud a (el muelle no trabaja: r = cte)')
print('(ii) a ≫ b :  ω² →', sp.limit(om2_sym, a_, sp.oo), '   ->  oscilador armónico del muelle (movimiento casi horizontal)')

#Valores numéricos
pars = (m, a, b, k, g)
print('\n--- Valores numéricos ---')
print(f"θ = +π/2 (arriba):  V'' = {d2V_num(np.pi/2, *pars):.4f}  ->  {'estable' if d2V_num(np.pi/2, *pars) > 0 else 'inestable'}")
print(f"θ = −π/2 (abajo) :  V'' = {d2V_num(-np.pi/2, *pars):.4f}  ->  estable")
om = np.sqrt(d2V_num(-np.pi/2, *pars)/(m*a**2))
print(f'ω (θ = −π/2) = {om:.5f} rad/s   (T = {2*np.pi/om:.5f} s)')
if a > b and k*(a**2 - b**2) > m*g*b:
    th_eq = np.arcsin(m*g*b/(k*(a**2 - b**2)))
    print(f'Equilibrios inestables: θ_eq = {th_eq:.5f} rad = {np.degrees(th_eq):.3f}°  y  π − θ_eq = {np.degrees(np.pi - th_eq):.3f}°   (V = {V_num(th_eq, *pars):.4f} J)')
    print(f"   V''(θ_eq) = {d2V_num(th_eq, *pars):.4f}  <  0  ->  inestable")
else:
    th_eq = None
    print('No existe otro equilibrio (k(a²−b²) ≤ m g b o a = b)')

## FUNCION PARA EL MOVIMIENTO (RUNGE-KUTTA 4)

In [ ]:
def F(y, pars):
    """Segundo miembro de las ecuaciones de Hamilton. y: (2, ncasos) = (θ, p). Los parámetros pueden ser arrays (uno por caso)."""
    d = f_num(*y, *pars)
    return np.array([np.broadcast_to(v, y.shape[1:]) for v in d])


def Motion(y0, pars, t):
    """Integra las ecuaciones de Hamilton con Runge-Kutta de orden 4 (varios casos a la vez)."""
    nv, nc = y0.shape
    N = len(t)
    y = np.zeros((nv, nc, N))
    y[:, :, 0] = y0

    for i in range(N - 1):
        dt = t[i+1] - t[i]
        yi = y[:, :, i]

        k1 = F(yi, pars)
        k2 = F(yi + 0.5*dt*k1, pars)
        k3 = F(yi + 0.5*dt*k2, pars)
        k4 = F(yi + dt*k3, pars)

        y[:, :, i+1] = yi + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

    return y


#Estado inicial: p = ∂L/∂θ̇ = m μ(θ) θ̇
th_0  = np.array([cs['th'] for cs in casos])
thd_0 = np.array([cs['thd'] for cs in casos])
y0 = np.array([th_0, m*mu_num(th_0, *pars)*thd_0])

#Simulación
y = Motion(y0, pars, t)
th, p = y
thd = qd_num(th, p, *pars)

## ENERGÍA, HAMILTONIANO Y DIAGRAMA DE FASES

In [ ]:
#Energía, Hamiltoniano y Lagrangiano
E = E_num(th, thd, *pars)                                  # E = T + U
H = H_num(th, p, *pars)                                    # H(θ, p)
L = L_num(th, thd, *pars)

print(f'max |H − E| / |E0|  (todos los casos) = {np.abs(H - E).max()/np.abs(E[:, 0]).max():.2e}')
print(f'max |H − H0| / |H0| (todos los casos) = {(np.abs(H - H[:, :1]).max(axis=1)/np.abs(H[:, 0])).max():.2e}   ->  E = H se conserva')

ig, ip = i_gen, i_peq
cols = {ip: 'tab:blue', ig: 'tab:red'}
th_g = np.linspace(-np.pi, np.pi, 600)

# --- Potencial V(θ) y niveles de energía ---
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(th_g, V_num(th_g, *pars), 'k', lw=2)
for i_, nm in ((ip, 'pequeñas oscilaciones'), (ig, 'movimiento general')):
    ax[0].axhline(E[i_, 0], color=cols[i_], ls='--', label=f'E ({nm}) = {E[i_, 0]:.2f} J')
ax[0].scatter([-np.pi/2], [V_num(-np.pi/2, *pars)], s=80, color='tab:green', zorder=5, label='equilibrio estable')
if d2V_num(np.pi/2, *pars) > 0:
    ax[0].scatter([np.pi/2], [V_num(np.pi/2, *pars)], s=80, color='tab:green', zorder=5)
else:
    ax[0].scatter([np.pi/2], [V_num(np.pi/2, *pars)], s=80, color='tab:red', marker='x', zorder=5, label='equilibrio inestable')
if th_eq is not None:
    ax[0].scatter([th_eq, np.pi - th_eq], [V_num(th_eq, *pars)]*2, s=80, color='tab:red', marker='x', zorder=5,
                  label=None if d2V_num(np.pi/2, *pars) <= 0 else 'equilibrio inestable')
ax[0].set(xlabel='θ (rad)', ylabel='V(θ) (J)', title='Energía potencial (muelle + gravedad)'); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

# --- Diagrama de fases (θ, p): curvas de nivel de H + trayectorias ---
pmax = 1.15*np.abs(p[[ip, ig]]).max()
THg, Pg = np.meshgrid(th_g, np.linspace(-pmax, pmax, 500))
Hg = H_num(THg, Pg, *pars)
ax[1].contour(THg, Pg, Hg, levels=30, colors='gainsboro', linewidths=0.8)
for i_ in (ip, ig):
    ax[1].contour(THg, Pg, Hg, levels=[E[i_, 0]], colors=cols[i_], linewidths=1, linestyles='--')
    thw = (th[i_] + np.pi) % (2*np.pi) - np.pi                              # θ envuelto a [−π, π]
    pw  = p[i_].copy(); thw = thw.astype(float)
    brk = np.where(np.abs(np.diff(thw)) > np.pi)[0] + 1                   # saltos por el envoltorio
    thw, pw = np.insert(thw, brk, np.nan), np.insert(pw, brk, np.nan)
    ax[1].plot(thw, pw, color=cols[i_], lw=2, label=casos[i_]['label'])
    ax[1].scatter(thw[0], pw[0], s=60, color=cols[i_], edgecolors='black', zorder=5)
ax[1].scatter([-np.pi/2], [0], s=80, color='tab:green', zorder=5)
ax[1].set(xlabel='θ (rad)', ylabel='p (kg m²/s)', title='Diagrama de fases (curvas de nivel de H)', xlim=(-np.pi, np.pi)); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# --- Conservación en el tiempo (caso general) ---
s = max(1, N // 20000)
fig, ax = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
ax[0].plot(t[::s], E[ig, ::s], color='tab:blue', lw=3, label='E = T + U')
ax[0].plot(t[::s], H[ig, ::s], 'r--', lw=1.5, label='H(θ, p)')
ax[0].plot(t[::s], L[ig, ::s], color='tab:purple', lw=1.2, label='L = T − U  (no se conserva)')
ax[0].set(ylabel='J', title='Caso general: E, H y L'); ax[0].legend(fontsize=8)
ax[1].plot(t[::s], th[ig, ::s], color='tab:gray', lw=1.5, label='θ (rad)')
ax[1].plot(t[::s], p[ig, ::s], color='tab:orange', lw=2, label='p  (no se conserva)')
ax[1].set(ylabel='θ , p', title='Coordenada y momento'); ax[1].legend(fontsize=8)
eps = 1e-18
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - H[ig, 0])/abs(H[ig, 0]) + eps, color='tab:red', lw=1, label='|H − H₀| / |H₀|  (integrador RK4)')
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - E[ig, ::s])/abs(E[ig, 0]) + eps, color='tab:green', lw=1, label='|H − E| / |E₀|  (redondeo)')
ax[2].set(xlabel='t (s)', ylabel='error relativo', title='Error numérico'); ax[2].legend(fontsize=8, loc='lower right')
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## (c) PEQUEÑAS OSCILACIONES: SOLUCIÓN ANALÍTICA Y CASOS LÍMITE

In [ ]:
#(c) Pequeñas oscilaciones alrededor de θ = −π/2:   θ(t) = −π/2 + δ0 cos(ω t)
delta0s  = np.pi/2 + th_0[:4]                                   # δ0 de los cuatro casos de pequeñas oscilaciones
delta    = th[:4] + np.pi/2                                     # δ(t) = θ + π/2
delta_an = delta0s[:, None]*np.cos(om*t)[None, :]

print(f'ω = sqrt[ g b/a² + (k/m)(1 − b²/a²) ] = {om:.5f} rad/s')
print(f'\n{"δ0":>8}{"max|δ−δ_an|/δ0":>18}')
for i_ in range(4):
    print(f'{delta0s[i_]:>8.3f}{np.abs(delta[i_] - delta_an[i_]).max()/delta0s[i_]:>18.2e}')

s = max(1, N // 5000)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(t[::s], th[i_peq, ::s], lw=2, label='numérica')
ax[0].plot(t[::s], (-np.pi/2 + delta_an[i_peq])[::s], 'r--', lw=1.2, label='−π/2 + δ0 cos(ω t)')
ax[0].axhline(-np.pi/2, color='k', ls=':', lw=1, label='θ_eq = −π/2')
ax[0].set(xlabel='t (s)', ylabel='θ (rad)', title='Pequeñas oscilaciones (δ0 = %.2f)' % delta0s[i_peq]); ax[0].legend(fontsize=8)
ax[1].plot(t[::s], (delta[i_peq] - delta_an[i_peq])[::s], color='tab:green')
ax[1].set(xlabel='t (s)', ylabel='δ_num − δ_an', title='Diferencia (desfase por no linealidad)')
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()


#Casos límite: (i) a = b  y  (ii) a ≫ b      (frecuencia numérica frente a la fórmula)
ratios = np.array([1.0, 1.5, 3.0, 10.0, 30.0])
a_arr  = ratios*b                                               # un valor de 'a' por caso (el resto de parámetros es común)
t_s    = np.arange(0, 12, 1e-3)
y0_s   = np.array([np.full(len(a_arr), -np.pi/2 + 0.05), np.zeros(len(a_arr))])
th_s_  = Motion(y0_s, (m, a_arr, b, k, g), t_s)[0]

def omega_numerica(x, tt):
    """Frecuencia a partir de los cruces ascendentes de x(t) por cero (interpolación lineal)."""
    cr = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]
    tc = tt[cr] - x[cr]*(tt[cr+1] - tt[cr])/(x[cr+1] - x[cr])
    return 2*np.pi/np.mean(np.diff(tc))

print(f'\n{"a/b":>6}{"ω teórica":>12}{"ω numérica":>13}{"sqrt(g/a)":>12}{"sqrt(k/m)":>12}')
for j, ra in enumerate(ratios):
    aj = a_arr[j]
    om_t = np.sqrt((m*g*b + k*(aj**2 - b**2))/(m*aj**2))
    om_n = omega_numerica(th_s_[j] + np.pi/2, t_s)
    print(f'{ra:>6.1f}{om_t:>12.4f}{om_n:>13.4f}{np.sqrt(g/aj):>12.4f}{np.sqrt(k/m):>12.4f}')
print('(i) a = b  ->  ω = sqrt(g/a) (péndulo simple)  ;  (ii) a ≫ b  ->  ω → sqrt(k/m) (oscilador del muelle)')

## ANIMACIÓN

In [ ]:
#Animación: función que anima un caso (elipse con muelle + diagrama de fases) y se aplica a las dos situaciones iniciales
from IPython.display import display
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 15       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 1.5      # duración de la estela de la partícula (s); 0 = sin estela
n_coils = 14       # vueltas del muelle dibujado

def spring_2d(x1, y1, n=n_coils, amp=0.06*b):
    """Zigzag que une el origen O con (x1, y1)."""
    d = max(np.hypot(x1, y1), 1e-9)
    u = np.array([x1, y1])/d                       # vector unitario a lo largo del muelle
    nrm = np.array([-u[1], u[0]])                  # normal
    sgrid = np.linspace(0, 1, 2*n + 1)
    off = amp*np.where(np.arange(2*n + 1) % 2 == 1, 1.0, -1.0)
    off[0] = off[-1] = 0                           # extremos rectos
    pts = np.outer(sgrid*d, u) + np.outer(off, nrm)
    return pts[:, 0], pts[:, 1]


def animate_ellipse(k_caso, titulo='', T_anim=T_anim, fps=fps, speed=speed, trail_s=trail_s):
    """Crea la figura y la animación del caso 'k_caso' (índice en 'casos'). Devuelve (fig, anim)."""
    step    = max(1, round(speed / (fps * dt)))
    idx     = np.arange(0, min(N, int(T_anim / dt)), step)
    trail_n = int(trail_s / (dt * step))

    ts, ths, ps_ = t[idx], th[k_caso, idx], p[k_caso, idx]
    xs, ys = a*np.cos(ths), b*np.sin(ths)
    thw = (ths + np.pi) % (2*np.pi) - np.pi                         # θ envuelto a [−π, π] para el diagrama de fases

    phi = np.linspace(0, 2*np.pi, 300)
    fig, (A1, A2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.4, 1]})
    fig.suptitle(titulo, fontsize=12)

    # --- Elipse con el muelle ---
    A1.plot(a*np.cos(phi), b*np.sin(phi), 'k-', lw=2)
    A1.axhline(0, color='gray', lw=0.5); A1.axvline(0, color='gray', lw=0.5); A1.plot(0, 0, 'ko', ms=5)
    A1.scatter([0], [-b], s=70, color='tab:green', zorder=3, label='equilibrio estable')
    if d2V_num(np.pi/2, *pars) > 0:
        A1.scatter([0], [b], s=70, color='tab:green', zorder=3)
    else:
        A1.scatter([0], [b], s=70, color='tab:red', marker='x', zorder=3, label='equilibrio inestable')
    if th_eq is not None:
        A1.scatter([a*np.cos(th_eq), a*np.cos(np.pi - th_eq)], [b*np.sin(th_eq)]*2, s=70, color='tab:red', marker='x', zorder=3,
                   label=None if d2V_num(np.pi/2, *pars) <= 0 else 'equilibrio inestable')
    A1.annotate('', xy=(1.25*a, -0.6*b), xytext=(1.25*a, 0.6*b), arrowprops=dict(arrowstyle='-|>', lw=2))
    A1.text(1.28*a, 0.0, 'g', fontsize=12)
    A1.set(xlim=(-1.4*a, 1.45*a), ylim=(-1.5*b, 1.5*b), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Partícula sobre la elipse')
    A1.legend(loc='upper left', fontsize=8); A1.grid(alpha=0.3)

    # --- Diagrama de fases ---
    pm = 1.15*np.abs(ps_).max() + 1e-9
    THg, Pg = np.meshgrid(np.linspace(-np.pi, np.pi, 300), np.linspace(-pm, pm, 300))
    A2.contour(THg, Pg, H_num(THg, Pg, *pars), levels=30, colors='gainsboro', linewidths=0.8)
    A2.contour(THg, Pg, H_num(THg, Pg, *pars), levels=[E[k_caso, 0]], colors='tab:red', linewidths=1, linestyles='--')
    A2.set(xlim=(-np.pi, np.pi), ylim=(-pm, pm), xlabel='θ (rad)', ylabel='p (kg m²/s)', title='Espacio de fases'); A2.grid(alpha=0.3)

    # Artistas
    spring, = A1.plot([], [], '-', color='dimgray', lw=1.3)
    bob,    = A1.plot([], [], 'o', color='tab:red', mec='black', ms=11, zorder=5)
    trail,  = A1.plot([], [], '-', color='tab:red', lw=1.2, alpha=0.6)
    dotP,   = A2.plot([], [], 'o', color='tab:red', mec='black', ms=9, zorder=5)
    trailP, = A2.plot([], [], '-', color='tab:red', lw=2)
    time_txt = A1.text(0.02, 0.05, '', transform=A1.transAxes)

    def update(k):
        j = max(0, k - trail_n)
        spring.set_data(*spring_2d(xs[k], ys[k]))
        bob.set_data([xs[k]], [ys[k]])
        trail.set_data(xs[j:k+1], ys[j:k+1])
        dotP.set_data([thw[k]], [ps_[k]])
        seg_t, seg_p = thw[j:k+1].astype(float), ps_[j:k+1].copy()
        brk = np.where(np.abs(np.diff(seg_t)) > np.pi)[0] + 1              # no unir los saltos por el envoltorio
        trailP.set_data(np.insert(seg_t, brk, np.nan), np.insert(seg_p, brk, np.nan))
        time_txt.set_text(f't = {ts[k]:.2f} s    θ = {np.degrees(thw[k]):.1f}°    r = {np.hypot(xs[k], ys[k]):.3f} m')
        return spring, bob, trail, dotP, trailP, time_txt

    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
    return fig, anim


# Se aplica a las dos situaciones iniciales
situaciones = [(i_gen, 'Situación 1: movimiento general'),
               (i_peq, 'Situación 2: pequeñas oscilaciones alrededor de θ = −π/2')]
anims = [animate_ellipse(k_caso, titulo) for k_caso, titulo in situaciones]

for fig_, anim_ in anims:
    plt.close(fig_)
    display(HTML(anim_.to_jshtml()))